# Lesson 20: Large Language Models, Perplexity & Google AI Studio Free Tier

In this notebook, you will:
1. Run open-source **GPT-2** text generation and compute **Perplexity** locally on Colab.
2. Perform **Zero-Shot** and **Few-Shot Prompt Engineering** using **Google AI Studio Free Tier** (`google-genai` + `GEMINI_API_KEY`).

In [ ]:
!pip install -q transformers google-genai
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
inputs = tok("Artificial intelligence will help developers", return_tensors="pt")
out = gpt2.generate(**inputs, max_new_tokens=25, do_sample=True, temperature=0.8)
print("GPT-2 Completion:", tok.decode(out[0], skip_special_tokens=True))

In [ ]:
# Compute Perplexity on a sample sentence with GPT-2
with torch.no_grad():
    loss = gpt2(**inputs, labels=inputs["input_ids"]).loss
print(f"Sample Perplexity: {torch.exp(loss).item():.2f}")

In [ ]:
# Few-Shot Prompting with Google AI Studio Free Tier (GEMINI_API_KEY)
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    import os
    api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    from google import genai
    client = genai.Client(api_key=api_key)
    few_shot_prompt = (
        "Extract the framework and task as JSON.\n"
        "Text: We trained a ResNet in PyTorch for image classification.\n"
        "JSON: {\"framework\": \"PyTorch\", \"task\": \"image classification\"}\n\n"
        "Text: We fine-tuned DistilBERT using HuggingFace Transformers for NER.\n"
        "JSON:"
    )
    resp = client.models.generate_content(model="gemini-2.5-flash", contents=few_shot_prompt)
    print(resp.text)
else:
    print("Add GEMINI_API_KEY to Colab Secrets to run live Gemini prompts.")